# LLM Prediction Split / Metrics 

## Imports and Config

In [9]:
import pandas as pd
import os
import json
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_curve, auc
)

## Data Splitting : (1) CSV with targets (for comparison) (2) CSV without targets (for LLM prompts)

In [12]:
# Load the cleaned CSV files
tabular_path = r"C:\Users\nehan\Documents\GT\CS 4641\ML_vs_LLM_COVID_GenderBias\Data\Cleaned Covid Data.csv"
prompt_path = r"C:\Users\nehan\Documents\GT\CS 4641\ML_vs_LLM_COVID_GenderBias\Data\Cleaned Prompt Covid Data.csv"

# Output directories
base_output_dir = r"C:\Users\nehan\Documents\GT\CS 4641\ML_vs_LLM_COVID_GenderBias\Data\LLM_Test_Splits_By_Target"
with_target_dir = os.path.join(base_output_dir, "WithTargets")
without_target_dir = os.path.join(base_output_dir, "WithoutTargets")
os.makedirs(with_target_dir, exist_ok=True)
os.makedirs(without_target_dir, exist_ok=True)

# Load data 
df_tabular = pd.read_csv(tabular_path)
df_prompt = pd.read_csv(prompt_path)

# Join prompt text with tabular data 
df_combined = df_tabular.join(df_prompt[["PATIENT DATAPOINT PROMPT"]])

# Max rows per output
max_rows = 10_000

# Helper function to sample up to max_rows
def sample_with_cap(df, cap=max_rows):
    return df.sample(n=min(len(df), cap), random_state=42)

# Splitting based on gender with sampling
gender_splits = {
    "AllMale": sample_with_cap(df_combined[df_combined["SEX"] == 2]),
    "AllFemale": sample_with_cap(df_combined[df_combined["SEX"] == 1]),
    "AllFemalePregnant": sample_with_cap(df_combined[(df_combined["SEX"] == 1) & (df_combined["PREGNANT"] == 1)]),
    "GenderBalanced": df_combined.groupby("SEX").apply(lambda x: x.sample(min(len(x), max_rows // 2), random_state=42)).reset_index(drop=True),
    "GenderUnbalanced": sample_with_cap(df_combined)
}

# Target columns 
target_cols = ["COVID-19 PRESENCE", "COVID-19 SEVERITY", "DEATH", "PNEUMONIA"]

# Saving
for gender_label, df_group in gender_splits.items():
    for target in target_cols:
        drop_targets = [col for col in target_cols if col != target]
        df_with = df_group.drop(columns=drop_targets)

        # Save version WITH target values
        with_target_path = os.path.join(with_target_dir, f"{gender_label}_{target.replace(' ', '')}_WithTargets.csv")
        df_with.to_csv(with_target_path, index=False)

        # Save version WITHOUT target values (but with header intact)
        df_without = df_with.copy()
        df_without[target] = pd.NA
        without_target_path = os.path.join(without_target_dir, f"{gender_label}_{target.replace(' ', '')}_WithoutTargets.csv")
        df_without.to_csv(without_target_path, index=False)

C:\Users\nehan\AppData\Local\Temp\ipykernel_22168\4290610658.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "GenderBalanced": df_combined.groupby("SEX").apply(lambda x: x.sample(min(len(x), max_rows // 2), random_state=42)).reset_index(drop=True),


## LLM Output Metrics


### Metric Computation Helper Function


In [3]:
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report
)
from collections import Counter
import json

def calculate_metrics(y_true, y_pred, target):
    # Detect if we're doing multiclass
    is_multiclass = target == "Covid-19Severity"

    # Accuracy and F1
    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Confusion matrix
    if is_multiclass:
        labels = [0, 1, 2, 3]
    else:
        labels = [0, 1]

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Label distribution
    true_counts = dict(Counter(y_true))
    pred_counts = dict(Counter(y_pred))
    all_labels = sorted(set(true_counts.keys()).union(set(pred_counts.keys())))
    distribution_diff = {
        str(label): {
            "actual_count": true_counts.get(label, 0),
            "predicted_count": pred_counts.get(label, 0),
            "difference": abs(true_counts.get(label, 0) - pred_counts.get(label, 0))
        }
        for label in all_labels
    }

    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "confusion_matrix": cm.tolist(),
        "label_distribution_comparison": distribution_diff,
        "total_actual": len(y_true),
        "total_predicted": len(y_pred),
        "dataset": "Test",
        "model_name": "Chatgpt"
    }

In [4]:
import pandas as pd
import os

# Prepare directories
with_targets_dir = r"C:\Users\nehan\Documents\GT\CS 4641\ML_vs_LLM_COVID_GenderBias\Data\LLM_Test_Splits_By_Target\WithTargets_gpt"
llm_predictions_dir = r"C:\Users\nehan\Documents\GT\CS 4641\ML_vs_LLM_COVID_GenderBias\Data\LLM_Test_Splits_By_Target\WithTargets_LLMPrediction_gpt"
output_dir = r"C:\Users\nehan\Documents\GT\CS 4641\ML_vs_LLM_COVID_GenderBias\Metric Output Files\LLM"
os.makedirs(output_dir, exist_ok=True)

# Mapping for gender distributions and target values
gender_keys = [
    "AllMale", "AllFemale", "GenderBalanced",
    "AllFemalePregnant", "GenderUnbalanced"
]
target_values = [
    "Covid-19Presence", "Covid-19Severity", "Death", "Pneumonia"
]

# Matching and metric evaluation logic
def match_files_and_evaluate():
    metrics_results = []

    for gender in gender_keys:
        for target in target_values:
            # Normalize target segment (e.g., Covid-19Presence → COVID-19PRESENCE)
        
            if "Covid-19" in target:
            # "Covid-19Presence" → "COVID-19 PRESENCE"
                target_col = "COVID-19 " + target[len("Covid-19"):].upper()
            else:
                target_col = target.upper()

            actual_file = f"{gender}_{target}_WithTargets.csv"
            llm_file = f"Chatgpt_{gender}_{target}_LLMTargets.csv"

            actual_path = os.path.join(with_targets_dir, actual_file)
            llm_path = os.path.join(llm_predictions_dir, llm_file)

            if not (os.path.exists(actual_path) and os.path.exists(llm_path)):
                print(f"Missing file(s): {actual_file} or {llm_file}")
                continue

            df_actual = pd.read_csv(actual_path)
            df_llm = pd.read_csv(llm_path)
            
            # Align by shortest row count
            min_len = min(len(df_actual), len(df_llm))
            df_actual = df_actual.iloc[:min_len]
            df_llm = df_llm.iloc[:min_len]


            # Normalize column names to uppercase
            df_actual.columns = df_actual.columns.str.upper()
            df_llm.columns = df_llm.columns.str.upper()

            if target_col not in df_actual.columns or target_col not in df_llm.columns:
                print(f"Skipping {gender}-{target}: Column '{target_col}' not found")
                continue

            if target == "Covid-19Severity":
                y_true = df_actual[target_col].astype(int)
                y_pred = df_llm[target_col].astype(int)
            else:
                y_true = (df_actual[target_col].astype(int) == 1).astype(int)
                y_pred = (df_llm[target_col].astype(int) == 1).astype(int)

            metrics = calculate_metrics(y_true, y_pred, target)

            # Save JSON file with metrics
            output_filename = f"{gender}_{target}_metrics.json"
            with open(os.path.join(output_dir, output_filename), "w") as f:
                json.dump(metrics, f, indent=4)

            metrics_results.append((gender, target, metrics))

    return metrics_results

# === RUN THE SCRIPT ===
results = match_files_and_evaluate()

# Save metrics summary
summary_df = pd.DataFrame(results, columns=["GenderDistribution", "TargetColumn", "Metrics"])
summary_df.to_csv("LLM_vs_True_Metrics_Summary.csv", index=False)

# Also display if in notebook
summary_df

,GenderDistribution,TargetColumn,Metrics
0,AllMale,Covid-19Presence,"{'accuracy': 0.5963452767972349, 'f1_macro': 0..."
1,AllMale,Covid-19Severity,"{'accuracy': 0.09564393572460932, 'f1_macro': ..."
2,AllMale,Death,"{'accuracy': 0.9077368382304506, 'f1_macro': 0..."
3,AllMale,Pneumonia,"{'accuracy': 0.853499149478644, 'f1_macro': 0...."
4,AllFemale,Covid-19Presence,"{'accuracy': 0.6432521358441725, 'f1_macro': 0..."
5,AllFemale,Covid-19Severity,"{'accuracy': 0.0076389443871594, 'f1_macro': 0..."
6,AllFemale,Death,"{'accuracy': 0.8930430833344704, 'f1_macro': 0..."
7,AllFemale,Pneumonia,"{'accuracy': 0.8966939490750035, 'f1_macro': 0..."
8,GenderBalanced,Covid-19Presence,"{'accuracy': 0.54688, 'f1_macro': 0.5327564904..."
9,GenderBalanced,Covid-19Severity,"{'accuracy': 0.51799, 'f1_macro': 0.2216310091..."
